# T33 - 信用债规模

## 第5章：策略实现与分析

---

### 分析模块概述

本章节实现规模分析的核心策略，包括：
1. 规模趋势分析
2. 结构分析
3. 集中度分析
4. 增长率分析

In [ ]:
# 导入依赖
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text, create_engine
from datetime import datetime, timedelta
import sqlalchemy.pool as pool
import os

# 从环境变量获取配置
MYSQL_HOST = os.environ.get('MYSQL_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
MYSQL_PORT = os.environ.get('MYSQL_PORT', '3306')
MYSQL_USER = os.environ.get('MYSQL_USER', 'hz_work')
MYSQL_PASSWORD = os.environ.get('MYSQL_PASSWORD', '')
MYSQL_DATABASE = os.environ.get('MYSQL_DATABASE', 'bond')

# 创建数据库连接
connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
sql_engine = create_engine(connection_string, poolclass=pool.NullPool)

def log_progress(message):
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{current_time}] {message}")

log_progress("数据库连接创建成功")

### 1. 规模趋势分析

In [ ]:
class ScaleTrendAnalyzer:
    """规模趋势分析器"""
    
    def __init__(self, engine):
        self.engine = engine
    
    def get_daily_scale(self, category_type='大类', category='全部', 
                         start_date=None, end_date=None):
        """
        获取日度规模数据
        
        Parameters:
        -----------
        category_type : str
            分类方式
        category : str
            分类名称
        start_date : str
            开始日期
        end_date : str
            结束日期
        
        Returns:
        --------
        DataFrame
        """
        sql = """
        SELECT DT, 余额
        FROM bond.信用债规模
        WHERE 分类方式 = :category_type AND 分类 = :category
        """
        
        params = {'category_type': category_type, 'category': category}
        
        if start_date:
            sql += " AND DT >= :start_date"
            params['start_date'] = start_date
        
        if end_date:
            sql += " AND DT <= :end_date"
            params['end_date'] = end_date
        
        sql += " ORDER BY DT"
        
        with self.engine.connect() as conn:
            result = pd.read_sql(text(sql), conn, params=params)
        
        result['DT'] = pd.to_datetime(result['DT'])
        result.set_index('DT', inplace=True)
        
        return result
    
    def analyze_trend(self, data, short_window=20, long_window=60):
        """
        分析规模趋势
        
        Parameters:
        -----------
        data : DataFrame
            日度规模数据
        short_window : int
            短期移动平均窗口
        long_window : int
            长期移动平均窗口
        
        Returns:
        --------
        dict
        """
        # 移动平均
        ma_short = data['余额'].rolling(short_window).mean()
        ma_long = data['余额'].rolling(long_window).mean()
        
        # 增长率
        growth_rate = data['余额'].pct_change() * 100
        
        # 加速度（增长率的二阶导）
        acceleration = growth_rate.diff()
        
        # 趋势识别
        if len(ma_short) > 5 and not ma_short.iloc[-5:].isna().all():
            recent_short = ma_short.iloc[-5:].mean()
            current_short = ma_short.iloc[-1]
            if current_short > recent_short * 1.01:
                trend = '上升'
            elif current_short < recent_short * 0.99:
                trend = '下降'
            else:
                trend = '稳定'
        else:
            trend = '数据不足'
        
        return {
            'ma_short': ma_short,
            'ma_long': ma_long,
            'growth_rate': growth_rate,
            'acceleration': acceleration,
            'trend': trend,
            'current_scale': data['余额'].iloc[-1] if len(data) > 0 else None,
            'avg_growth_rate': growth_rate.mean()
        }

# 创建分析器实例
trend_analyzer = ScaleTrendAnalyzer(sql_engine)
log_progress("趋势分析器初始化完成")

In [ ]:
# 执行趋势分析示例
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

# 获取全部债券规模趋势
total_scale = trend_analyzer.get_daily_scale(
    category_type='大类',
    category='全部',
    start_date=start_date,
    end_date=end_date
)

print(f"数据记录数: {len(total_scale)}")
if len(total_scale) > 0:
    trend_result = trend_analyzer.analyze_trend(total_scale)
    print(f"\n趋势分析结果:")
    print(f"  当前趋势: {trend_result['trend']}")
    print(f"  当前规模: {trend_result['current_scale']:.2f} 亿元")
    print(f"  平均增长率: {trend_result['avg_growth_rate']:.4f}%")

### 2. 结构分析

In [ ]:
class StructureAnalyzer:
    """结构分析器"""
    
    def __init__(self, engine):
        self.engine = engine
    
    def get_structure(self, dt, category_type):
        """
        获取指定日期的结构数据
        
        Parameters:
        -----------
        dt : str
            查询日期
        category_type : str
            分类方式
        
        Returns:
        --------
        DataFrame
        """
        sql = """
        SELECT 分类, 子类, 余额
        FROM bond.信用债规模
        WHERE DT = :dt AND 分类方式 = :category_type
        ORDER BY 余额 DESC
        """
        
        params = {'dt': str(dt), 'category_type': category_type}
        
        with self.engine.connect() as conn:
            result = pd.read_sql(text(sql), conn, params=params)
        
        return result
    
    def analyze_structure(self, dt):
        """
        分析指定日期的结构特征
        
        Parameters:
        -----------
        dt : str
            查询日期
        
        Returns:
        --------
        dict
        """
        analysis = {}
        
        # 1. 大类结构
        major_structure = self.get_structure(dt, '大类')
        if len(major_structure) > 0:
            total = major_structure['余额'].sum()
            major_structure['占比'] = major_structure['余额'] / total * 100
            analysis['major_structure'] = major_structure
            analysis['dominant_type'] = major_structure.iloc[0]['分类']
            analysis['concentration_ratio'] = major_structure.iloc[0]['占比']
        
        # 2. 评级结构
        rating_structure = self.get_structure(dt, '评级')
        if len(rating_structure) > 0:
            rating_total = rating_structure['余额'].sum()
            rating_structure['占比'] = rating_structure['余额'] / rating_total * 100
            analysis['rating_structure'] = rating_structure
            
            # 高评级占比
            high_rating = rating_structure[
                rating_structure['子类'].isin(['AAA', 'AA+', 'AA'])
            ]['余额'].sum()
            analysis['high_rating_ratio'] = high_rating / rating_total * 100 if rating_total > 0 else 0
        
        # 3. 久期结构
        term_structure = self.get_structure(dt, '久期')
        if len(term_structure) > 0:
            term_total = term_structure['余额'].sum()
            term_structure['占比'] = term_structure['余额'] / term_total * 100
            analysis['term_structure'] = term_structure
            
            # 短期占比
            short_term = term_structure[
                term_structure['子类'].isin([0.5, 1, 2])
            ]['余额'].sum()
            analysis['short_term_ratio'] = short_term / term_total * 100 if term_total > 0 else 0
        
        return analysis

# 创建分析器实例
structure_analyzer = StructureAnalyzer(sql_engine)
log_progress("结构分析器初始化完成")

In [ ]:
# 执行结构分析示例
# 获取最新日期
with sql_engine.connect() as conn:
    latest_dt = pd.read_sql("SELECT MAX(DT) as max_dt FROM bond.信用债规模", conn)
    latest_date = latest_dt['max_dt'].iloc[0]

print(f"分析日期: {latest_date}")

structure_result = structure_analyzer.analyze_structure(latest_date)

print("\n=== 大类结构 ===")
if 'major_structure' in structure_result:
    print(structure_result['major_structure'][['分类', '余额', '占比']].to_string())
    print(f"\n主导类型: {structure_result['dominant_type']}")
    print(f"集中度: {structure_result['concentration_ratio']:.2f}%")

print("\n=== 评级结构 ===")
if 'rating_structure' in structure_result:
    print(structure_result['rating_structure'][['子类', '余额', '占比']].head(10).to_string())
    print(f"\n高评级占比: {structure_result['high_rating_ratio']:.2f}%")

### 3. 集中度分析

In [ ]:
class ConcentrationAnalyzer:
    """集中度分析器"""
    
    def __init__(self, engine):
        self.engine = engine
    
    def calculate_hhi(self, shares):
        """
        计算HHI指数（赫芬达尔-赫希曼指数）
        
        Parameters:
        -----------
        shares : array-like
            市场份额（百分比形式）
        
        Returns:
        --------
        float
        """
        # 将百分比转换为小数
        shares_decimal = np.array(shares) / 100
        hhi = np.sum(shares_decimal ** 2)
        return hhi
    
    def calculate_cr(self, shares, n=10):
        """
        计算CRn（前n大占比）
        
        Parameters:
        -----------
        shares : array-like
            市场份额
        n : int
            前n大
        
        Returns:
        --------
        float
        """
        sorted_shares = np.sort(shares)[::-1]
        return np.sum(sorted_shares[:n])
    
    def analyze_concentration(self, dt, category_type):
        """
        分析集中度
        
        Parameters:
        -----------
        dt : str
            查询日期
        category_type : str
            分类方式
        
        Returns:
        --------
        dict
        """
        sql = """
        SELECT 分类, 子类, 余额
        FROM bond.信用债规模
        WHERE DT = :dt AND 分类方式 = :category_type
        ORDER BY 余额 DESC
        """
        
        params = {'dt': str(dt), 'category_type': category_type}
        
        with self.engine.connect() as conn:
            data = pd.read_sql(text(sql), conn, params=params)
        
        if len(data) == 0:
            return None
        
        total = data['余额'].sum()
        data['占比'] = data['余额'] / total * 100
        
        shares = data['占比'].values
        
        return {
            'hhi': self.calculate_hhi(shares),
            'cr5': self.calculate_cr(shares, 5),
            'cr10': self.calculate_cr(shares, 10),
            'n_items': len(data),
            'top5': data.head(5)[['分类', '子类', '余额', '占比']],
            'concentration_level': self._get_concentration_level(self.calculate_hhi(shares))
        }
    
    def _get_concentration_level(self, hhi):
        """判断集中度级别"""
        if hhi > 0.3:
            return '高度集中'
        elif hhi > 0.2:
            return '中度集中'
        elif hhi > 0.1:
            return '低度集中'
        else:
            return '分散'

# 创建分析器实例
concentration_analyzer = ConcentrationAnalyzer(sql_engine)
log_progress("集中度分析器初始化完成")

In [ ]:
# 执行集中度分析示例
print(f"分析日期: {latest_date}")

# 分析申万一级集中度
print("\n=== 申万一级行业集中度 ===")
sw_concentration = concentration_analyzer.analyze_concentration(latest_date, '申万一级')
if sw_concentration:
    print(f"HHI指数: {sw_concentration['hhi']:.4f}")
    print(f"CR5: {sw_concentration['cr5']:.2f}%")
    print(f"CR10: {sw_concentration['cr10']:.2f}%")
    print(f"集中度级别: {sw_concentration['concentration_level']}")
    print("\n前5大行业:")
    print(sw_concentration['top5'].to_string())

### 4. 增长率分析

In [ ]:
class GrowthAnalyzer:
    """增长率分析器"""
    
    def __init__(self, engine):
        self.engine = engine
    
    def get_growth_data(self, category_type='大类', category='全部', 
                         start_date=None, end_date=None):
        """
        获取增长率数据
        
        Parameters:
        -----------
        category_type : str
            分类方式
        category : str
            分类名称
        start_date : str
            开始日期
        end_date : str
            结束日期
        
        Returns:
        --------
        DataFrame
        """
        sql = """
        SELECT DT, 余额
        FROM bond.信用债规模
        WHERE 分类方式 = :category_type AND 分类 = :category
        """
        
        params = {'category_type': category_type, 'category': category}
        
        if start_date:
            sql += " AND DT >= :start_date"
            params['start_date'] = start_date
        
        if end_date:
            sql += " AND DT <= :end_date"
            params['end_date'] = end_date
        
        sql += " ORDER BY DT"
        
        with self.engine.connect() as conn:
            data = pd.read_sql(text(sql), conn, params=params)
        
        if len(data) == 0:
            return data
        
        data['DT'] = pd.to_datetime(data['DT'])
        data.set_index('DT', inplace=True)
        
        return data
    
    def analyze_growth(self, data, window=30):
        """
        分析增长率
        
        Parameters:
        -----------
        data : DataFrame
            日度规模数据
        window : int
            滚动窗口
        
        Returns:
        --------
        dict
        """
        if len(data) < 2:
            return None
        
        # 日增长率
        data['growth_rate'] = data['余额'].pct_change() * 100
        
        # 滚动平均增长率
        data['ma_growth'] = data['growth_rate'].rolling(window).mean()
        
        # 增长加速度
        data['acceleration'] = data['growth_rate'].diff()
        
        # 增长阶段识别
        latest = data.iloc[-1]
        latest_growth = latest['growth_rate']
        latest_acc = latest['acceleration']
        
        if pd.isna(latest_growth) or pd.isna(latest_acc):
            stage = '数据不足'
        elif latest_growth > 0 and latest_acc > 0:
            stage = '加速增长'
        elif latest_growth > 0 and latest_acc < 0:
            stage = '减速增长'
        elif latest_growth < 0 and latest_acc < 0:
            stage = '加速下降'
        elif latest_growth < 0 and latest_acc > 0:
            stage = '减速下降'
        else:
            stage = '稳定'
        
        return {
            'data': data,
            'avg_growth_rate': data['growth_rate'].mean(),
            'max_growth_rate': data['growth_rate'].max(),
            'min_growth_rate': data['growth_rate'].min(),
            'growth_stage': stage,
            'latest_growth': latest_growth if not pd.isna(latest_growth) else None,
            'latest_acceleration': latest_acc if not pd.isna(latest_acc) else None
        }

# 创建分析器实例
growth_analyzer = GrowthAnalyzer(sql_engine)
log_progress("增长率分析器初始化完成")

In [ ]:
# 执行增长率分析示例
print(f"分析日期范围: {start_date} ~ {end_date}")

growth_data = growth_analyzer.get_growth_data(
    category_type='大类',
    category='全部',
    start_date=start_date,
    end_date=end_date
)

if len(growth_data) > 0:
    growth_result = growth_analyzer.analyze_growth(growth_data)
    
    if growth_result:
        print("\n=== 增长率分析 ===")
        print(f"平均增长率: {growth_result['avg_growth_rate']:.4f}%")
        print(f"最大增长率: {growth_result['max_growth_rate']:.4f}%")
        print(f"最小增长率: {growth_result['min_growth_rate']:.4f}%")
        print(f"当前增长阶段: {growth_result['growth_stage']}")
        print(f"最新日增长率: {growth_result['latest_growth']:.4f}%" if growth_result['latest_growth'] else "最新日增长率: N/A")

### 综合分析报告

In [ ]:
def generate_analysis_report(dt):
    """
    生成综合分析报告
    
    Parameters:
    -----------
    dt : str
        分析日期
    
    Returns:
    --------
    dict
    """
    report = {
        'date': dt,
        'structure': structure_analyzer.analyze_structure(dt),
        'major_concentration': concentration_analyzer.analyze_concentration(dt, '大类'),
        'rating_concentration': concentration_analyzer.analyze_concentration(dt, '评级'),
        'term_concentration': concentration_analyzer.analyze_concentration(dt, '久期')
    }
    
    return report

# 生成报告
print(f"生成综合分析报告: {latest_date}")
report = generate_analysis_report(latest_date)

print("\n=== 综合分析报告 ===")
print(f"分析日期: {report['date']}")

if report['structure'].get('dominant_type'):
    print(f"\n主导大类: {report['structure']['dominant_type']}")
    print(f"集中度: {report['structure']['concentration_ratio']:.2f}%")
    print(f"高评级占比: {report['structure'].get('high_rating_ratio', 0):.2f}%")
    print(f"短期占比: {report['structure'].get('short_term_ratio', 0):.2f}%")

---

### 分析模块完成

**已实现的分析功能**:
1. 规模趋势分析 - 移动平均、增长率、趋势识别
2. 结构分析 - 大类结构、评级结构、久期结构
3. 集中度分析 - HHI指数、CR5/CR10、集中度级别
4. 增长率分析 - 滚动增长率、加速度、增长阶段

---

**下一章**: [6_可视化与报告](6_可视化与报告.ipynb)